In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 6.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://strode-clothes-crested.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://strode-clothes-crested.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, MHUST）是一所位於台灣新竹縣新豐鄉的科技大學。它以其務實致用、與產業緊密結合的辦學特色而聞名，尤其在新竹科學園區周邊的產業人才培育上扮演重要角色。

以下是明新科技大學的簡要介紹：

1.  **歷史沿革**：
    *   學校創立於1966年，前身為「明新工業專科學校」。
    *   隨著台灣技職教育體系的發展，於1997年升格為「明新技術學院」。
    *   最終於2002年改制為「明新科技大學」，成為一所綜合型的科技大學。

2.  **地理位置與產業連結**：
    *   明新科大地理位置優越，毗鄰全球知名的**新竹科學園區**，以及湖口工業區等。
    *   這一地理優勢使得學校與周邊的高科技產業、傳統產業及服務業連結非常緊密，為學生提供了豐富的實習與就業機會。
    *   學校也因此能快速掌握產業脈動，調整課程內容以符合業界需求，培養符合市場趨勢的人才。

3.  **教育理念與特色**：
    *   **務實致用**：學校強調理論與實務並重，課程設計著重於應用性，設有大量的實驗室、實習工廠和專業教室，讓學生在動手實作中學習。
    *   **產學合作**：積極推動與企業的產學合作計畫，包括共同開發技術、提供學生實習機會、開設產業專班等，確保教學內容與產業接軌。
    *   **創新創業**：鼓勵學生發揮創意，參與創新專題競賽，並提供創業輔導資源。
    *   **全人教育**：在培養專業技能的同時，也重視學生的品德教育、人文素養和國際視野的拓展。

4.  **學院與學系**：
    明新科大設有多元化的學院與學系，主要涵蓋以下領域：
    *   **工程學院**：包含電機、電子、機械、資工、化工等，是學校的傳統強項。
    *   **管理學院**：包含企管、資管、財金、行銷、休憩等，培養具備管理與商業實務能力的專業人才。
    *   **服務事業學院**：包含旅館管理、餐飲管理、幼兒保育、老人服務事業管理等，因應社會服務業需求。
    *   **人文社會學院**：提供通識教育及相關系所。

5.  **畢業生表現**：
    明新科大的畢業生因其紮實的專業技能和良好的實作經驗，在

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學的現任校長是 **劉國偉** 教授。

劉國偉校長於2022年8月1日就任明新科技大學第12任校長。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 💡【0702 核心功能：有狀態對話判斷 (Stateful Control)】
        # 當使用者發送的訊息是以 "AI "（大寫 AI 加上一個半形空格）開頭時
        if text.startswith('AI '):
            prompt = text[3:]
            # 💡【為什麼這裡的回應是 Stateful？】
            # 這裡呼叫的是 stateful_query(prompt)，其底層運行的是 chat.send_message(message=payload)。
            # 因為使用了 client.chats.create() 建立的 chat 物件，它會將每一次的對話記錄保留在記憶體中。
            # 這使得機器人具備了「記憶能力」，能夠理解上下文（Context）。
            # 例如：第一句傳送『AI 簡介明新科技大學』，第二句傳送『AI 校長是誰？』，
            # 機器人能自動從歷史紀錄中知道『誰』指的是『明新科技大學的校長』，這就是標準的有狀態（Stateful）多輪對話。
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )
 # 💡【學舌鸚鵡機制：保持無狀態】
            # 若不是以 "AI " 開頭，則單純將使用者傳入的文字複製兩份發送回去，不影響且不記入 Gemini 的聊天歷史。
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information abo

BODY:  {"destination":"U07d9297fdd48ee689d0e289df2083566","events":[{"type":"message","message":{"type":"text","id":"616645681213341960","quoteToken":"GmBOai87SsNrbxFBSV0W7n5NvXkMcl0nVjNpWLc1jIIyGD0znNm3RPwurB8Ze0VmqD45J3Nxf0d5C_Dh_ivIkasKpt3b-TAdB-VU6JyNsLb3_NqWZwNJ2jjdBnIipbdmsmz8rfOOHaZ-YF7MHB1Jiw","markAsReadToken":"cIzGGrXKoQsTxSBbP_glvKwt_oGWs5NSiI3-LVM0ts8fE750DRmWpTcMY8HRVdqEk-x2wnAEVJKdtWg4VzEXZiYCG4ta6re-e7LXDKAZ2-m2uCk1iKwfoLbQyXZFikq2Ztgj86YDOVsK1bonL4Chcq3RTjzJr9A4lRrCyHIdADki2qBXt8mcLmNCXxlTb6lXgngn6yr2S1Ad2Zi_SlGtOA","text":"Hello"},"webhookEventId":"01KT3FD9YWE0JN513SNQVK7DRK","deliveryContext":{"isRedelivery":false},"timestamp":1780380771955,"source":{"type":"user","userId":"U437fda835928fa6b4a10144ab18a651f"},"replyToken":"f9d8ab823c4443018dda22ed0ad7dc57","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 06:13:20] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U07d9297fdd48ee689d0e289df2083566","events":[{"type":"message","message":{"type":"text","id":"616645728005521686","quoteToken":"SdpO2F1mxs-uMTN8kPJnemnJkXakbqKYSZQ1h7KxQYIn42r5ejiuBotz0j1DMG8pm53tYnpl98EmAbggcZt5KMBfuD1TJ9Rb7Fv1kFf2U8fNkPqnio_WKnl_aEQ2sLpiz-Uy5nQxGH1bflVBE870Hg","markAsReadToken":"urr_47WLIK2_HroEXuI24zwTwEnWub3wBpxZ8UVIR_3VTASdcHOc2gIH1aPUXO52jgdwny6koqjKn_gMsDMrlSwQ4ffTRwLoyty5gH7rtFB-GQKJsZKZs6udxsy2XZFS-qE1g3XKQJMAIBjGmL0JnJH_mx_TYoBY5NlI7XOBJ82tv5URtFk8cOm_KqYu4dfo6G-iOCJ1LSPX9YfyUx9dhA","text":"簡介明新科技大學"},"webhookEventId":"01KT3FE5A0TQAPHYS70QAQQR00","deliveryContext":{"isRedelivery":false},"timestamp":1780380799869,"source":{"type":"user","userId":"U437fda835928fa6b4a10144ab18a651f"},"replyToken":"5b5c98aa482e4434a90dc66248615273","mode":"active"}]}
